In [ ]:
#!pip install torch transformers datasets pandas scikit-learn openpyxl accelerate
#!pip install sentence-transformers

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier

# =====================
# Load data
# =====================
df = pd.read_excel("/content/Annotated_Data.xlsx")

df["text"] = df["target_tweet"].astype(str) + " " + df["authentic_reply"].astype(str)

label_columns = ["STANCE", "ACTION", "PERSONALNESS", "POLITENESS"]

X = df["text"]
Y = df[label_columns]


In [ ]:
# =====================
# Encode labels
# =====================
encoders = {}
Y_encoded = pd.DataFrame()

for col in label_columns:
    le = LabelEncoder()
    Y_encoded[col] = le.fit_transform(Y[col])
    encoders[col] = le

# =====================
# Train / Val / Test split
# =====================
X_train, X_temp, Y_train, Y_temp = train_test_split(
    X, Y_encoded, test_size=0.30, random_state=42
)

X_val, X_test, Y_val, Y_test = train_test_split(
    X_temp, Y_temp, test_size=0.50, random_state=42
)

In [ ]:
# =====================
# TF-IDF Embedding
# =====================
#vectorizer = TfidfVectorizer(max_features=15000, ngram_range=(1,2), stop_words="english")

#X_train_vec = vectorizer.fit_transform(X_train)
#X_val_vec = vectorizer.transform(X_val)
#X_test_vec = vectorizer.transform(X_test)

In [ ]:
#Sbert Embedding
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("all-MiniLM-L6-v2")

X_train_vec = embedder.encode(X_train.tolist(), show_progress_bar=True)
X_val_vec = embedder.encode(X_val.tolist())
X_test_vec = embedder.encode(X_test.tolist())


In [ ]:
#xgboost
xgb_base = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="multi:softmax",
    eval_metric="mlogloss",
    tree_method="hist"
)

In [ ]:
model = MultiOutputClassifier(xgb_base)

# =====================
# Train
# =====================
model.fit(X_train_vec, Y_train)

MultiOutputClassifier(estimator=XGBClassifier(base_score=None, booster=None,
                                              callbacks=None,
                                              colsample_bylevel=None,
                                              colsample_bynode=None,
                                              colsample_bytree=0.8, device=None,
                                              early_stopping_rounds=None,
                                              enable_categorical=False,
                                              eval_metric='mlogloss',
                                              feature_types=None,
                                              feature_weights=None, gamma=None,
                                              grow_policy=None,
                                              importance_type=None,
                                              interaction_constraints=None,
                                              learning_rate=0.1, max_bin=None,
                                              max_cat_threshold=None,
                                              max_cat_to_onehot=None,
                                              max_delta_step=None, max_depth=6,
                                              max_leaves=None,
                                              min_child_weight=None,
                                              missing=nan,
                                              monotone_constraints=None,
                                              multi_strategy=None,
                                              n_estimators=300, n_jobs=None,
                                              num_parallel_tree=None, ...))

In [ ]:
preds = model.predict(X_test_vec)

In [ ]:
print("\nTest accuracy per label:")
for i, col in enumerate(label_columns):
    acc = accuracy_score(Y_test.iloc[:, i], preds[:, i])
    print(f"{col}: {acc:.4f}")


Test accuracy per label:
STANCE: 0.5000
ACTION: 0.7083
PERSONALNESS: 0.6000
POLITENESS: 0.5250


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/18 [00:00<?, ?it/s]

In [ ]:
 #Logistic Regression
 base_lr = LogisticRegression(
    max_iter=3000,
    n_jobs=-1,
    class_weight="balanced"
)

model = MultiOutputClassifier(base_lr)

In [ ]:
model.fit(X_train_vec, Y_train)

# =====================
# Evaluate
# =====================
preds = model.predict(X_test_vec)

In [ ]:
print("\nTest accuracy per label:")
for i, col in enumerate(label_columns):
    acc = accuracy_score(Y_test.iloc[:, i], preds[:, i])
    print(f"{col}: {acc:.4f}")


Test accuracy per label:
STANCE: 0.4333
ACTION: 0.4833
PERSONALNESS: 0.4417
POLITENESS: 0.3417


In [ ]:
#Linear SVC

from sklearn.svm import LinearSVC

model = MultiOutputClassifier(
    LinearSVC(class_weight="balanced")
)

In [ ]:
model.fit(X_train_vec, Y_train)


MultiOutputClassifier(estimator=LinearSVC(class_weight='balanced'))

In [ ]:
preds= model.predict(X_test_vec)

In [ ]:
print("\nTest accuracy per label:")
for i, col in enumerate(label_columns):
    acc = accuracy_score(Y_test.iloc[:, i], preds[:, i])
    print(f"{col}: {acc:.4f}")


Test accuracy per label:
STANCE: 0.4250
ACTION: 0.5750
PERSONALNESS: 0.5167
POLITENESS: 0.3500


In [ ]:
#Training deberta model

import pandas as pd
import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModel,
    Trainer,
    TrainingArguments
)
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

MODEL_NAME = "microsoft/deberta-v3-small"
DATA_PATH = "Annotated_Data.xlsx"
LABEL_COLS = ["STANCE", "ACTION", "PERSONALNESS", "POLITENESS"]
MAX_LEN = 256

# ---------------------------
# Load data
# ---------------------------
df = pd.read_excel(DATA_PATH)
df["text"] = df["target_tweet"].astype(str) + " [SEP] " + df["authentic_reply"].astype(str)

# Encode labels
encoders = {}
label_arrays = []

for col in LABEL_COLS:
    le = LabelEncoder()
    enc = le.fit_transform(df[col])
    encoders[col] = le
    label_arrays.append(enc)

Y = np.vstack(label_arrays).T

X_train, X_test, Y_train, Y_test = train_test_split(
    df["text"].tolist(), Y, test_size=0.2, random_state=42
)

# ---------------------------
# Dataset
# ---------------------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class MultiTaskDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=MAX_LEN,
            return_tensors="pt"
        )

        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

train_ds = MultiTaskDataset(X_train, Y_train)
test_ds = MultiTaskDataset(X_test, Y_test)

# ---------------------------
# Model
# ---------------------------
class MultiTaskDeberta(nn.Module):
    def __init__(self, model_name, num_classes):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden = self.encoder.config.hidden_size

        self.heads = nn.ModuleList([
            nn.Linear(hidden, n) for n in num_classes
        ])

    def forward(self, input_ids=None, attention_mask=None, labels=None):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        pooled = outputs.last_hidden_state[:, 0]

        logits = [head(pooled) for head in self.heads]

        loss = None
        if labels is not None:
            loss_fn = nn.CrossEntropyLoss()
            loss = sum(
                loss_fn(logits[i], labels[:, i])
                for i in range(len(logits))
            )

        return {"loss": loss, "logits": logits}

num_classes = [len(encoders[c].classes_) for c in LABEL_COLS]
model = MultiTaskDeberta(MODEL_NAME, num_classes)

# ---------------------------
# Metrics
# ---------------------------
def compute_metrics(eval_pred):
    preds, labels = eval_pred

    accs = {}
    for i, col in enumerate(LABEL_COLS):
        p = np.argmax(preds[i], axis=1)
        accs[f"{col}_acc"] = accuracy_score(labels[:, i], p)

    accs["avg_acc"] = np.mean(list(accs.values()))
    return accs

# Custom prediction wrapper
class MultiTaskTrainer(Trainer):
    def prediction_step(self, model, inputs, prediction_loss_only=False, ignore_keys=None):
        with torch.no_grad():
            outputs = model(**inputs)
            loss = outputs["loss"]
            logits = outputs["logits"]

        logits = [l.detach().cpu().numpy() for l in logits]
        labels = inputs["labels"].detach().cpu().numpy()

        return (loss.detach().cpu().numpy(), logits, labels)

# ---------------------------
# Training
# ---------------------------
args = TrainingArguments(
    output_dir="./deberta_multitask",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    weight_decay=0.01,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="avg_acc",
    fp16=torch.cuda.is_available()
)

trainer = MultiTaskTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    compute_metrics=compute_metrics
)

trainer.train()

# ---------------------------
# Final evaluation
# ---------------------------
metrics = trainer.evaluate()
print("\nFinal metrics:")
for k, v in metrics.items():
    print(k, ":", v)

# Save model
trainer.save_model("./deberta_multitask_final")
tokenizer.save_pretrained("./deberta_multitask_final")

print("\nModel saved to ./deberta_multitask_final")


tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/578 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:566: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


pytorch_model.bin:   0%|          | 0.00/286M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/286M [00:00<?, ?B/s]

TypeError: TrainingArguments.__init__() got an unexpected keyword argument 'evaluation_strategy'